# 17 — Semi-Autoregressive Drafting

**Engineering statement:** semi-autoregressive drafting specifies schedulable prefixes.

Notebook 17 connects the confidence scheduler from Notebook 13 to the draft model that supplies its inputs. The scheduler can only allocate verification effectively when the drafter produces prefixes with useful survival structure. DSpark's semi-autoregressive design keeps a parallel backbone for speed while adding a lightweight sequential dependency mechanism to improve suffix quality.

## Start Here

```text
00: decoding creates latency pressure
07: verification becomes an engineering resource
13: confidence schedules that resource
17: draft generation shapes the schedulable prefixes
```

Notebook flow:

```text
prompt
→ draft blocks
→ semi-autoregressive generation
→ confidence profile
→ scheduled verification
→ accepted prefix
→ throughput
```

In [ ]:
from pathlib import Path
import json
import math
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, display

# Robust paths whether run from repository root, notebooks/, or an uploaded copy.
CWD = Path.cwd().resolve()
if (CWD / "src").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "17_semi_autoregressive_drafting"
SITE = REPO_ROOT / "site" / "2606-19348"
SITE_IMAGES = SITE / "images"

for path in [NOTEBOOKS, FIGURES, RESULTS, SITE_IMAGES]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "font.size": 11,
})

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")

## 1. Why drafting matters

Notebook 13 treated confidence scores as the scheduling input. Notebook 17 asks where those scores come from.

A drafter is not only a fast auxiliary model. It creates the candidate block whose prefix survival curve determines how much verification can be scheduled profitably. If the suffix decays too quickly, the scheduler has little to allocate. If the drafter preserves local coherence, the scheduler receives longer useful prefixes.

The practical question becomes:

> How should a draft model generate tokens so that confidence scheduling has useful prefixes to schedule?

In [ ]:
# Figure 17_generation_pipeline.png
steps = ["prompt", "draft\nmodel", "candidate\nblock", "confidence\nprofile", "scheduler", "target\nverification", "accepted\ntokens"]
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.axis("off")
y = 0.5
xs = np.linspace(0.05, 0.95, len(steps))
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, y, label, ha="center", va="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.42", fc="white", ec="black", lw=1.5),
            transform=ax.transAxes)
    if i < len(steps) - 1:
        ax.annotate("", xy=(xs[i+1]-0.055, y), xytext=(x+0.055, y), xycoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("Draft generation creates the scheduler input", fontsize=18, pad=20)
fig.tight_layout()
path = FIGURES / "17_generation_pipeline.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 2. Block generation strategies

There are three useful abstractions for Notebook 17:

1. **Autoregressive drafting** conditions on previous draft tokens, but drafting latency grows with block length.
2. **Parallel drafting** proposes all positions together, but suffix quality decays because token dependencies inside the block are weak.
3. **Semi-autoregressive drafting** preserves most of the parallel speed while adding a lightweight sequential head that models local transitions.

This notebook does not reproduce DSpark training. It builds executable toy models that expose the scheduling consequences of these generation styles.

In [ ]:
@dataclass(frozen=True)
class DraftStrategy:
    name: str
    latency_base: float
    latency_per_token: float
    confidence: list

strategies = [
    DraftStrategy("autoregressive", latency_base=1.00, latency_per_token=0.12,
                  confidence=[0.74, 0.78, 0.80, 0.81, 0.81, 0.80, 0.79, 0.78]),
    DraftStrategy("parallel", latency_base=0.72, latency_per_token=0.01,
                  confidence=[0.90, 0.76, 0.60, 0.45, 0.32, 0.22, 0.15, 0.10]),
    DraftStrategy("semi-autoregressive", latency_base=0.76, latency_per_token=0.025,
                  confidence=[0.90, 0.84, 0.77, 0.68, 0.58, 0.48, 0.38, 0.30]),
]

def prefix_survival(conf):
    return np.cumprod(np.asarray(conf, dtype=float))

def expected_accepted(conf, ell=None, bonus=True):
    survival = prefix_survival(conf)
    if ell is None:
        ell = len(survival)
    value = survival[:ell].sum()
    return value + (1 if bonus else 0)

def draft_latency(strategy, block_len=8):
    return strategy.latency_base + strategy.latency_per_token * block_len

strategy_table = pd.DataFrame([
    {
        "strategy": s.name,
        "draft_latency_units": draft_latency(s),
        "expected_accepted_full_block": expected_accepted(s.confidence, bonus=False),
        "position_1_confidence": s.confidence[0],
        "position_8_confidence": s.confidence[-1],
    }
    for s in strategies
])
strategy_table

In [ ]:
# Figure 17_block_generation.png
fig, ax = plt.subplots(figsize=(10.5, 4.0))
ax.axis("off")
rows = ["autoregressive", "parallel", "semi-autoregressive"]
ys = [0.78, 0.50, 0.22]
labels = list("ABCDEFGH")
for row, y in zip(rows, ys):
    ax.text(0.02, y, row, ha="left", va="center", fontsize=12, fontweight="bold", transform=ax.transAxes)
    for i, tok in enumerate(labels):
        x = 0.25 + i * 0.075
        if row == "autoregressive":
            fill = "white"
            hatch = None
            edge = "black"
        elif row == "parallel":
            fill = "#f2f4f7"
            hatch = "//"
            edge = "0.55"
        else:
            fill = "white" if i == 0 else "#f2f4f7"
            hatch = None if i < 3 else ".."
            edge = "black" if i < 5 else "0.55"
        rect = plt.Rectangle((x, y-0.065), 0.055, 0.13, transform=ax.transAxes,
                             fc=fill, ec=edge, lw=1.3, hatch=hatch)
        ax.add_patch(rect)
        ax.text(x+0.0275, y, tok, ha="center", va="center", fontsize=11, transform=ax.transAxes)
    if row == "autoregressive":
        ax.text(0.88, y, "dependent but slower", ha="center", va="center", fontsize=11, transform=ax.transAxes)
    elif row == "parallel":
        ax.text(0.88, y, "fast but suffix decays", ha="center", va="center", fontsize=11, transform=ax.transAxes)
    else:
        ax.text(0.88, y, "fast + local dependencies", ha="center", va="center", fontsize=11, transform=ax.transAxes)
ax.set_title("Block generation strategies", fontsize=18, pad=18)
fig.tight_layout()
path = FIGURES / "17_block_generation.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 3. Confidence profiles

The scheduler sees a confidence profile. The drafter creates that profile.

A pure parallel drafter may have high first-position confidence and rapid suffix decay. An autoregressive drafter may keep suffix quality more stable, but with higher drafting latency. A semi-autoregressive drafter seeks the useful middle: high early confidence, slower suffix decay, and modest extra latency.

In [ ]:
# Figure 17_confidence_profiles.png
positions = np.arange(1, 9)
fig, ax = plt.subplots(figsize=(8.8, 5.0))
for s in strategies:
    ax.plot(positions, s.confidence, marker="o", label=s.name)
ax.set_xlabel("Draft position")
ax.set_ylabel("Conditional confidence $c_j$")
ax.set_title("Drafting strategy shapes the confidence profile")
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
path = FIGURES / "17_confidence_profiles.png"
fig.savefig(path)
plt.show()
path

## 4. Prefix survival and schedulable length

Confidence scheduling uses prefix survival, not isolated confidence.

\[
a_j=\prod_{i=1}^{j} c_i
\]

Even moderate conditional-confidence differences compound quickly across a prefix. The semi-autoregressive draft profile therefore matters because small improvements at later positions can create much longer schedulable prefixes.

In [ ]:
# Figure 17_prefix_survival.png
fig, ax = plt.subplots(figsize=(8.8, 5.0))
for s in strategies:
    ax.plot(positions, prefix_survival(s.confidence), marker="o", label=s.name)
ax.axhline(0.25, ls="--", lw=1.2, label="example scheduling threshold 0.25")
ax.set_xlabel("Draft position")
ax.set_ylabel("Prefix survival $a_j$")
ax.set_title("Small suffix-quality gains create longer schedulable prefixes")
ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
path = FIGURES / "17_prefix_survival.png"
fig.savefig(path)
plt.show()
path

In [ ]:
def scheduled_length(conf, threshold=0.25):
    return int(np.sum(prefix_survival(conf) >= threshold))

prefix_gain = pd.DataFrame([
    {
        "strategy": s.name,
        "scheduled_length_threshold_025": scheduled_length(s.confidence, 0.25),
        "expected_accepted_scheduled": expected_accepted(s.confidence, scheduled_length(s.confidence, 0.25), bonus=False),
        "draft_latency_units": draft_latency(s),
    }
    for s in strategies
])
prefix_gain

In [ ]:
# Figure 17_prefix_gain.png
fig, ax = plt.subplots(figsize=(7.8, 4.6))
ax.bar(prefix_gain["strategy"], prefix_gain["scheduled_length_threshold_025"])
ax.set_ylabel("Schedulable prefix length")
ax.set_title("Semi-autoregression lengthens the schedulable prefix")
ax.tick_params(axis="x", rotation=10)
fig.tight_layout()
path = FIGURES / "17_prefix_gain.png"
fig.savefig(path)
plt.show()
path

## 5. Drafting cost versus useful prefix value

A drafter is useful when the accepted tokens gained from its proposals justify its latency. A semi-autoregressive design adds a small sequential component, so it is not free. The engineering question is whether that extra dependency modeling produces enough accepted-prefix value to improve end-to-end throughput.

This notebook uses a toy throughput proxy:

\[
\Theta_D = \frac{\tau_D}{T_{\mathrm{draft},D}+T_{\mathrm{verify}}}
\]

where the subscript \(D\) indicates that the confidence profile and latency depend on the drafting strategy.

In [ ]:
T_VERIFY = 1.40
rows = []
for s in strategies:
    ell = scheduled_length(s.confidence, 0.25)
    tau = expected_accepted(s.confidence, ell, bonus=True)
    lat = draft_latency(s) + T_VERIFY
    rows.append({
        "strategy": s.name,
        "scheduled_length": ell,
        "expected_accepted_tau": tau,
        "draft_latency": draft_latency(s),
        "round_latency": lat,
        "throughput_proxy": tau / lat,
    })
throughput_df = pd.DataFrame(rows)
throughput_df

In [ ]:
# Figure 17_drafting_vs_throughput.png
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(throughput_df["strategy"], throughput_df["throughput_proxy"])
ax.set_ylabel("Toy throughput proxy")
ax.set_title("Useful prefixes must justify drafting cost")
ax.tick_params(axis="x", rotation=10)
for i, row in throughput_df.iterrows():
    ax.text(i, row["throughput_proxy"] + 0.025, f"ℓ={int(row['scheduled_length'])}", ha="center")
fig.tight_layout()
path = FIGURES / "17_drafting_vs_throughput.png"
fig.savefig(path)
plt.show()
path

## 6. Semi-autoregressive architecture as a schedulability mechanism

DSpark's architectural idea can be read as a schedulability mechanism:

```text
parallel backbone
→ fast block proposal
→ lightweight sequential head
→ local dependency modeling
→ slower suffix decay
→ longer useful prefixes
```

The architecture is not merely a better drafter. It changes the verification budget available to the scheduler.

In [ ]:
# Figure 17_architecture_flow.png
steps = ["parallel\nbackbone", "base\nlogits", "lightweight\nsequential head", "local\ndependencies", "confidence\nprofile", "schedulable\nprefix"]
fig, ax = plt.subplots(figsize=(11, 2.9))
ax.axis("off")
xs = np.linspace(0.06, 0.94, len(steps))
y = 0.5
for i, (x, label) in enumerate(zip(xs, steps)):
    ax.text(x, y, label, ha="center", va="center", fontsize=12.5,
            bbox=dict(boxstyle="round,pad=0.42", fc="white", ec="black", lw=1.5),
            transform=ax.transAxes)
    if i < len(steps)-1:
        ax.annotate("", xy=(xs[i+1]-0.06, y), xytext=(x+0.06, y), xycoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("Semi-autoregressive drafting creates schedulable prefixes", fontsize=18, pad=20)
fig.tight_layout()
path = FIGURES / "17_architecture_flow.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 7. Decoding stack

Notebook 17 closes the loop between generation and scheduling. The full stack is now:

```text
prompt
→ drafter architecture
→ confidence profile
→ scheduler
→ target verification
→ accepted tokens
→ throughput
```

This prepares Notebook 23, which will treat throughput as the main optimization object rather than only a consequence of accepted prefix length.

In [ ]:
# Figure 17_decoding_stack.png
layers = [
    ("prompt", "input context"),
    ("drafter", "parallel backbone + sequential head"),
    ("confidence", "conditional scores and prefix survival"),
    ("scheduler", "allocate verification budget"),
    ("target", "parallel verification"),
    ("serving", "accepted tokens and throughput"),
]
fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.axis("off")
y_positions = np.linspace(0.86, 0.14, len(layers))
for i, ((name, desc), y) in enumerate(zip(layers, y_positions)):
    ax.text(0.24, y, name, ha="center", va="center", fontsize=13, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=1.5), transform=ax.transAxes)
    ax.text(0.56, y, desc, ha="left", va="center", fontsize=12, transform=ax.transAxes)
    if i < len(layers)-1:
        ax.annotate("", xy=(0.24, y_positions[i+1]+0.055), xytext=(0.24, y-0.055), xycoords=ax.transAxes,
                    arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("Confidence-scheduled decoding stack", fontsize=18, pad=16)
fig.tight_layout()
path = FIGURES / "17_decoding_stack.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path

## 8. Engineering questions

| Stage | Engineering question | Next notebook |
|---|---|---|
| Drafting | How does the drafter shape the confidence profile? | 23 — Throughput Objective |
| Prefix survival | How much suffix quality is needed for longer schedules? | 23 |
| Architecture | How much sequential dependency is worth the latency? | 23 |
| Scheduling | How does better drafting change the throughput optimum? | 29 — Hardware-Aware Scheduling |
| Serving | When should the system spend more compute on better drafts? | 37 — Operating Regimes |

## 9. Key equations

Draft block:

\[
D=(d_1,d_2,\ldots,d_B)
\]

Prefix survival:

\[
a_j=\prod_{i=1}^{j}c_i
\]

Expected accepted tokens:

\[
\tau_D(\ell)=1+\sum_{j=1}^{\ell}a_{D,j}
\]

Draft-dependent latency:

\[
T_D=T_{\mathrm{verify}}+T_{\mathrm{draft},D}
\]

Drafting objective:

\[
D^*=\arg\max_D\frac{\tau_D}{T_D}
\]

The scheduler allocates verification, but the drafter determines the prefix survival curve being allocated.

In [ ]:
# Save machine-readable outputs.
strategy_table.to_csv(RESULTS / "draft_strategy_summary.csv", index=False)
prefix_gain.to_csv(RESULTS / "prefix_gain.csv", index=False)
throughput_df.to_csv(RESULTS / "drafting_throughput_proxy.csv", index=False)

figures = [
    "17_generation_pipeline.png",
    "17_block_generation.png",
    "17_confidence_profiles.png",
    "17_prefix_survival.png",
    "17_prefix_gain.png",
    "17_drafting_vs_throughput.png",
    "17_architecture_flow.png",
    "17_decoding_stack.png",
]

manifest = {
    "notebook": "17_semi_autoregressive_drafting.ipynb",
    "title": "Semi-Autoregressive Drafting",
    "engineering_statement": "Semi-autoregressive drafting specifies schedulable prefixes.",
    "figures": figures,
    "results": [
        "draft_strategy_summary.csv",
        "prefix_gain.csv",
        "drafting_throughput_proxy.csv",
    ],
    "next_notebook": "23_throughput_objective.ipynb",
}
(RESULTS / "17_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Copy figures into site image folder if this repo layout is present.
for fig_name in figures:
    src = FIGURES / fig_name
    dst = SITE_IMAGES / fig_name
    if src.exists():
        shutil.copyfile(src, dst)

manifest

## 10. Repository roadmap

```text
00 Context
07 Verification Resource
13 Confidence Scheduling
17 Semi-Autoregressive Drafting
23 Throughput Objective
29 Hardware-Aware Scheduling
37 Operating Regimes
43 Resource Allocation
49 Adaptive AI Infrastructure
```

Notebook 17 completes the first algorithmic arc: a drafter creates candidate prefixes, confidence turns them into survival curves, and a scheduler allocates verification. Notebook 23 turns this into an explicit throughput objective.

## 11. Download artifacts

Run the final cell to package the Notebook 17 outputs for download.

In [ ]:
# Package Notebook 17 artifacts for download.
zip_path = RESULTS / "17_semi_autoregressive_drafting_artifacts.zip"

notebook_path = NOTEBOOKS / "17_semi_autoregressive_drafting.ipynb"
fallback_notebook_path = Path.cwd() / "17_semi_autoregressive_drafting.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    RESULTS,
]
for fig_name in figures:
    paths_to_include.append(FIGURES / fig_name)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file() or path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass